# Random Forest Combo Ablation: Remove Worst PNPLA3 Problem Slices

This notebook removes only the PNPLA3 concentration slices that repeatedly showed weak ranking or very large error on the final outer test, then retrains the same frozen Random Forest baseline.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns

project_root = Path.cwd().resolve()
while not (project_root / "utils").exists() and project_root != project_root.parent:
    project_root = project_root.parent

if not (project_root / "utils").exists():
    raise RuntimeError("Could not locate project root containing the utils package")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from classical_ml.random_forest import (
    RandomForestExperimentConfig,
    RandomForestHoldoutSplitter,
    RandomForestTrainer,
    compute_regression_metrics,
)
from classical_ml.random_forest.data import RandomForestDataBuilder
from utils.merge_historic_data import load_merged_dataset

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)


## Frozen Configuration

This uses the same strict-cleaning, add-mRNA, no-normalized-conditions RF setup and the same frozen Optuna hyperparameters as the final baseline notebook.

In [2]:
cmsirna_path = os.environ.get("CMSIRNA_RAW_DATA_PATH")
historic_path = os.environ.get("CMSIRNA_RAW_HISTORIC_DATA_PATH")

assert cmsirna_path, "CMSIRNA_RAW_DATA_PATH is not set"
assert historic_path, "CMSIRNA_RAW_HISTORIC_DATA_PATH is not set"

config = RandomForestExperimentConfig(
    target_column="Inhibition",
    strict_cleaning=True,
    add_mrna=True,
    use_normalized_conditions=False,
    outer_test_size=0.33,
    n_splits=3,
    leak_n=30,
    random_state=42,
    n_estimators=500,
    max_depth=None,
    min_samples_split=7,
    min_samples_leaf=1,
    max_features=0.3,
    n_jobs=-1,
)

baseline_metrics = pd.DataFrame([{'pearson': 0.287733, 'spearman': 0.335756, 'rmse': 36.207205, 'mae': 28.724077}])
config


RandomForestExperimentConfig(target_column='Inhibition', target_len=25, strict_cleaning=True, add_mrna=True, fetch_missing_mrna=True, use_normalized_conditions=False, n_splits=3, leak_n=30, outer_test_size=0.33, random_state=42, n_estimators=500, max_depth=None, min_samples_split=7, min_samples_leaf=1, max_features=0.3, n_jobs=-1, tuning_n_iter=20, tuning_scoring='neg_mean_absolute_error', optuna_n_trials=20)

## What This Notebook Removes

Removed combinations:
- `group == "PNPLA3" and Concentration_nM in {0.1, 0.5, 1.0}`

Why: PNPLA3 appears repeatedly in the weakest `Concentration + Gene` slices. This tests whether the main poisoning effect is coming from PNPLA3 inside the low-dose regime rather than from all PNPLA3 rows or all low-dose rows.

In [3]:
builder = RandomForestDataBuilder(config)
merged_df = load_merged_dataset(cmsirna_path, historic_path)
enriched_df = builder.pipeline.enrich_dataset_with_encodings(
    merged_df,
    strict_cleaning=config.strict_cleaning,
    add_mrna=config.add_mrna,
)

combo_mask = (
    (enriched_df["gene_target_symbol_name"] == "PNPLA3")
    & (enriched_df["Concentration_nM"].isin([0.1, 0.5, 1.0]))
)
removed_enriched_df = enriched_df.loc[combo_mask].copy()
filtered_enriched_df = enriched_df.loc[~combo_mask].reset_index(drop=True)

X, groups, y = builder.pipeline.prepare_for_classical_ml(
    filtered_enriched_df,
    target_column=config.target_column,
    use_normalized_conditions=config.use_normalized_conditions,
)

removal_summary = pd.DataFrame([{
    "starting_rows": int(len(enriched_df)),
    "rows_removed": int(len(removed_enriched_df)),
    "rows_remaining": int(len(filtered_enriched_df)),
    "pct_removed": float(len(removed_enriched_df) / len(enriched_df)),
    "removed_combo": "PNPLA3 @ {0.1, 0.5, 1.0} nM",
}])
removal_summary


loaded 3515 historic rows
merged 43153 CMsiRNA and 3515 historic rows into 46668
Running qc and data cleaning
dropped 4233 rows for in-vivo readings
dropped 565 rows for mM readings
dropped 115 rows for unknown or unwanted cell lines
dropped 47 rows for out-of-range inhibition
dropped 1749 rows for missing or unknown concentration
dropped 796 rows for concentration > 200 nM
filled 7522 rows for missing time of administration
dropped 2198 rows with a missing or >25 nt strand
dropped 6 columns: ['Modification_locations_Sense_strand', 'Modification_locations_Antisense_strand', 'Modifications_sense_strand', 'Modifications_AntiSense_strand_3_5', 'position_Antisense_strand', 'position_Sense_strand']
Mapping mRNA structural profiles
Error reading reference dataset: [Errno 2] No such file or directory: '/home/larsena8/software/fennec/src/fennec/support_files/train_data_v1.1.0_N=27742.csv'
Loaded 47 gene sequences from local cache
Loaded 0 gene sequences from reference CSV
Building gene -> mRNA

,starting_rows,rows_removed,rows_remaining,pct_removed,removed_combo
0,36965,3629,33336,0.098174,"PNPLA3 @ {0.1, 0.5, 1.0} nM"


## Refit And Evaluate

The model is retrained from scratch after the combo ablation and then evaluated again with the same outer-test holdout workflow.

In [4]:
holdout_splitter = RandomForestHoldoutSplitter(config)
train_idx, test_idx = holdout_splitter.split(X, y, groups)

X_train = X[train_idx]
y_train = y[train_idx]
groups_train = groups[train_idx]
X_test = X[test_idx]
y_test = y[test_idx]
groups_test = groups[test_idx]

trainer = RandomForestTrainer(config)
trainer.fit(X_train, y_train)
y_test_pred = trainer.predict(X_test)

metric_values = compute_regression_metrics(y_test, y_test_pred)
ablation_metrics = pd.DataFrame([metric_values])
split_summary = pd.DataFrame([{
    "n_train": int(len(train_idx)),
    "n_test": int(len(test_idx)),
    "n_train_groups": int(len(np.unique(groups_train))),
    "n_test_groups": int(len(np.unique(groups_test))),
}])

split_summary


,n_train,n_test,n_train_groups,n_test_groups
0,26101,7235,36,18


## Baseline Comparison

In [5]:
comparison = pd.concat(
    [baseline_metrics.assign(run="baseline_final_test"), ablation_metrics.assign(run="ablation")],
    ignore_index=True,
)
comparison


,pearson,spearman,rmse,mae,run
0,0.287733,0.335756,36.207205,28.724077,baseline_final_test
1,0.184268,0.234984,33.014696,25.476859,ablation


In [6]:
delta_vs_baseline = ablation_metrics.iloc[0] - baseline_metrics.iloc[0]
pd.DataFrame({"metric": delta_vs_baseline.index, "delta_vs_baseline": delta_vs_baseline.values})


,metric,delta_vs_baseline
0,pearson,-0.103465
1,spearman,-0.100772
2,rmse,-3.192509
3,mae,-3.247218


## Notes To Fill In After Running

- Did this focused removal help Spearman more than the broad ablations?
- Did error improve without throwing away too much data?
- Does the gain suggest a real poisoning hotspot or just ordinary hard biology?